In [ ]:
# Cell 1: imports, config, constants
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from statsmodels.tsa.stattools import acf
import statsmodels.api as sm
from scipy import stats

FILE_PATH = 'fichinH125.xlsx'   # adjust if needed

SESSION_START = pd.Timedelta('9:00:00')
SESSION_END   = pd.Timedelta('13:30:00')
BIN_MINUTES   = 15
N_BINS        = 18

BIN_LABELS = [
    pd.Timedelta(hours=9, minutes=15*i)
    for i in range(N_BINS)
]
BIN_LABEL_STRS = [f'{int(9 + 15*i//60):02d}:{int(15*i%60):02d}' for i in range(N_BINS)]

WEEKDAY_ORDER = ['lunes', 'martes', 'miércoles', 'jueves', 'viernes']

In [ ]:
# Cell 2: pure functions

def load_data(path):
    df = pd.read_excel(path, dtype={'Fecha': str, 'Tiempo': str})
    df['Fecha'] = pd.to_datetime(df['Fecha'], dayfirst=True)
    df['Tiempo_td'] = pd.to_timedelta(df['Tiempo'])
    df['Timestamp'] = df['Fecha'] + df['Tiempo_td']
    return df

def assign_bin(tiempo_td):
    # returns bin index 0..17 or -1 if outside session or exactly 13:30
    if tiempo_td < SESSION_START or tiempo_td >= SESSION_END:
        return -1
    offset = int((tiempo_td - SESSION_START).total_seconds() // 60)
    return offset // BIN_MINUTES

def build_bin_volume(df):
    # returns (session_df, bin_vol) where bin_vol has columns: date, bin_idx, bin_label, Monto
    df = df.copy()
    df['bin_idx'] = df['Tiempo_td'].apply(assign_bin)
    in_session = df[df['bin_idx'] >= 0]
    bv = (
        in_session.groupby(['Fecha', 'bin_idx'])['Monto']
        .sum()
        .reset_index()
    )
    # fill missing bins with 0
    all_dates = df['Fecha'].unique()
    idx = pd.MultiIndex.from_product([all_dates, range(N_BINS)], names=['Fecha', 'bin_idx'])
    bv = bv.set_index(['Fecha', 'bin_idx']).reindex(idx, fill_value=0).reset_index()
    bv['bin_label'] = bv['bin_idx'].apply(lambda i: BIN_LABEL_STRS[i])
    return bv

def compute_session_totals(bv):
    return bv.groupby('Fecha')['Monto'].sum().rename('V')

def compute_shares(bv, session_totals):
    bv = bv.copy()
    bv['V'] = bv['Fecha'].map(session_totals)
    bv['share'] = np.where(bv['V'] > 0, bv['Monto'] / bv['V'], 0.0)
    return bv

def profile_stats(bv_shares):
    g = bv_shares.groupby('bin_idx')['share']
    return pd.DataFrame({
        'bin_label': BIN_LABEL_STRS,
        'mean':   g.mean().values,
        'median': g.median().values,
        'q25':    g.quantile(0.25).values,
        'q75':    g.quantile(0.75).values,
    })

def weekday_profiles(bv_shares):
    bv_shares = bv_shares.copy()
    bv_shares['dow'] = bv_shares['Fecha'].dt.day_name(locale='es_ES.UTF-8') \
        if False else bv_shares['Fecha'].map(
            lambda d: ['lunes','martes','miércoles','jueves','viernes',None,None][d.weekday()]
        )
    return bv_shares.groupby(['dow', 'bin_idx'])['share'].mean().unstack(level=0)

def monthend_profiles(bv_shares):
    bv_shares = bv_shares.copy()
    bv_shares['monthend'] = bv_shares['Fecha'].dt.day >= 25
    return bv_shares.groupby(['monthend', 'bin_idx'])['share'].mean().unstack(level=0)

def compute_acf_series(series, nlags):
    vals, ci = acf(series, nlags=nlags, alpha=0.05, fft=True)
    return vals[1:], ci[1:]  # drop lag-0

def pool_within_session_acf(bv_shares, nlags=5):
    # compute u = v / (mu_hat * V), pool across sessions respecting boundaries
    mu = bv_shares.groupby('bin_idx')['share'].mean()
    bv = bv_shares.copy()
    bv['mu'] = bv['bin_idx'].map(mu)
    # u(t,d) = share / mu  (== v / (mu * V))
    bv['u'] = np.where(bv['mu'] > 0, bv['share'] / bv['mu'], np.nan)
    # for each session, extract the ordered 18-length u vector and compute lagged products
    dates = sorted(bv['Fecha'].unique())
    lag_products = {lag: [] for lag in range(1, nlags + 1)}
    for d in dates:
        u_sess = bv[bv['Fecha'] == d].sort_values('bin_idx')['u'].values
        for lag in range(1, nlags + 1):
            pairs = list(zip(u_sess[lag:], u_sess[:-lag]))
            lag_products[lag].extend(pairs)
    u_all = bv['u'].dropna().values
    u_mean = np.nanmean(u_all)
    u_var  = np.nanvar(u_all)
    acf_vals = {}
    for lag, pairs in lag_products.items():
        pairs = [(a, b) for a, b in pairs if not (np.isnan(a) or np.isnan(b))]
        if pairs:
            a_arr = np.array([p[0] for p in pairs])
            b_arr = np.array([p[1] for p in pairs])
            acf_vals[lag] = np.mean((a_arr - u_mean) * (b_arr - u_mean)) / u_var
        else:
            acf_vals[lag] = np.nan
    n_total = len(u_all)
    ci_bound = 1.96 / np.sqrt(n_total)
    return acf_vals, ci_bound

def fit_ar_regression(log_v_series):
    s = log_v_series.dropna().reset_index(drop=True)
    df_reg = pd.DataFrame({'y': s, 'lag1': s.shift(1), 'lag2': s.shift(2)})
    df_reg = df_reg.dropna()
    X = sm.add_constant(df_reg[['lag1', 'lag2']])
    res = sm.OLS(df_reg['y'], X).fit()
    return res

def flag_outlier_sessions(session_totals, bv_shares, sigma_thresh=2.5, share_thresh=0.30):
    log_v = np.log(session_totals)
    mu_lv, std_lv = log_v.mean(), log_v.std()
    flags = []
    for d, v in session_totals.items():
        z = (np.log(v) - mu_lv) / std_lv
        if abs(z) > sigma_thresh:
            flags.append({'date': d, 'reason': f'V outlier (z={z:.2f})', 'stat': v})
    max_share = bv_shares.groupby('Fecha')['share'].max()
    for d, ms in max_share.items():
        if ms > share_thresh:
            flags.append({'date': d, 'reason': f'bin share > {share_thresh} (max={ms:.3f})', 'stat': ms})
    return pd.DataFrame(flags).sort_values('date') if flags else pd.DataFrame(columns=['date','reason','stat'])

def print_summary(label, text):
    print(f'\n--- {label} ---')
    print(text)

In [ ]:
# Cell 3: execution

# ── load ──────────────────────────────────────────────────────────────────────
raw = load_data(FILE_PATH)
print('Loaded:', len(raw), 'rows')
print('Columns:', raw.columns.tolist())
print('Date range:', raw['Fecha'].min().date(), '→', raw['Fecha'].max().date())

# timestamp sanity check
by_date = raw.groupby('Fecha')['Tiempo_td'].agg(['min','max'])
outside = by_date[(by_date['min'] < SESSION_START) | (by_date['max'] > SESSION_END)]
print('\nTimestamp sanity — dates with trades outside 9:00–13:30:')
print(outside if len(outside) else '  none')

# Reg distribution
reg_counts = raw['Reg'].value_counts(dropna=False)
reg_share  = reg_counts / len(raw)
print('\nReg value counts and share:')
print(pd.DataFrame({'count': reg_counts, 'share': reg_share.round(4)}))
print('\nMonto stats by Reg:')
print(raw.groupby('Reg')['Monto'].describe().round(0))

# ── decision point: ask user before dropping R ────────────────────────────────
# Default: include all Reg values. Set DROP_R = True after inspecting output above.
DROP_R = False
working = raw[raw['Reg'] != 'R'].copy() if DROP_R else raw.copy()
print(f'\nDROP_R={DROP_R}  →  working set: {len(working)} rows')

# ── bin construction ──────────────────────────────────────────────────────────
dropped_1330 = (working['Tiempo_td'] == SESSION_END).sum()
print(f'Rows at exactly 13:30:00 (dropped): {dropped_1330}')

bv       = build_bin_volume(working)
sess_tot = compute_session_totals(bv)
bv_s     = compute_shares(bv, sess_tot)
print(f'Sessions: {sess_tot.shape[0]}  |  bins: {bv.shape[0]}')


# ══════════════════════════════════════════════════════════════════════════════
# D1. Average intraday volume profile
# ══════════════════════════════════════════════════════════════════════════════
prof = profile_stats(bv_s)

fig, ax = plt.subplots(figsize=(10, 4))
x = np.arange(N_BINS)
ax.bar(x, prof['mean'], color='steelblue', alpha=0.7, label='Mean share')
ax.fill_between(x, prof['q25'], prof['q75'], alpha=0.25, color='steelblue', label='IQR')
ax.plot(x, prof['median'], 'o-', color='black', lw=1.2, ms=4, label='Median')
ax.set_xticks(x)
ax.set_xticklabels(BIN_LABEL_STRS, rotation=45, ha='right', fontsize=8)
ax.set_ylabel('Share of daily volume')
ax.set_title('D1 — Average intraday volume profile')
ax.legend(fontsize=8)
ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1, decimals=1))
plt.tight_layout()
plt.savefig('d1_profile.png', dpi=120)
plt.show()

flat_share = 1 / N_BINS
max_dev_flat = (prof['mean'] - flat_share).abs().max()
print('\nD1 μ(t):')
print(prof[['bin_label','mean','median']].to_string(index=False))
print(f'Uniform share = {flat_share:.4f}  |  max |μ(t) - 1/18| = {max_dev_flat:.4f}')
shape_flag = 'visibly non-flat' if max_dev_flat > 0.01 else 'approximately flat'
print(f'Shape: {shape_flag}')


# ══════════════════════════════════════════════════════════════════════════════
# D2. Stability of diurnal shape
# ══════════════════════════════════════════════════════════════════════════════
wd_prof = weekday_profiles(bv_s)
me_prof = monthend_profiles(bv_s)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
x = np.arange(N_BINS)

ax = axes[0]
for wd in WEEKDAY_ORDER:
    if wd in wd_prof.columns:
        ax.plot(x, wd_prof[wd].values, marker='.', lw=1.2, ms=4, label=wd)
ax.set_xticks(x)
ax.set_xticklabels(BIN_LABEL_STRS, rotation=45, ha='right', fontsize=7)
ax.set_ylabel('Mean share')
ax.set_title('D2 — Profile by weekday')
ax.legend(fontsize=7)
ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1, decimals=1))

ax = axes[1]
cols_me = me_prof.columns.tolist()
labels_me = {True: 'month-end (day≥25)', False: 'non month-end'}
for col in cols_me:
    ax.plot(x, me_prof[col].values, marker='.', lw=1.5, ms=4, label=labels_me.get(col, str(col)))
ax.set_xticks(x)
ax.set_xticklabels(BIN_LABEL_STRS, rotation=45, ha='right', fontsize=7)
ax.set_ylabel('Mean share')
ax.set_title('D2 — Profile: month-end vs rest')
ax.legend(fontsize=7)
ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1, decimals=1))

plt.tight_layout()
plt.savefig('d2_stability.png', dpi=120)
plt.show()

# max abs deviation across weekdays
wd_arr = wd_prof[WEEKDAY_ORDER].values  # shape (18, 5)
wd_max_dev = np.nanmax(np.abs(wd_arr[:, :, None] - wd_arr[:, None, :])) if wd_arr.size else np.nan
me_arr = me_prof.values
me_max_dev = np.nanmax(np.abs(me_arr[:, 0] - me_arr[:, 1])) if me_arr.shape[1] == 2 else np.nan
print(f'\nD2 max absolute deviation between weekday shapes (any bin): {wd_max_dev:.4f}')
print(f'D2 max absolute deviation month-end vs non (any bin):       {me_max_dev:.4f}')


# ══════════════════════════════════════════════════════════════════════════════
# D3. Persistence of V(d)
# ══════════════════════════════════════════════════════════════════════════════
log_v = np.log(sess_tot)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

ax = axes[0]
ax.plot(log_v.values, lw=1, color='steelblue')
ax.set_xlabel('Session index')
ax.set_ylabel('log V(d)')
ax.set_title('D3 — log V(d) time series')

acf_vals_v, acf_ci_v = compute_acf_series(log_v.values, nlags=10)
lags_v = np.arange(1, 11)
ci_v = 1.96 / np.sqrt(len(log_v))

ax = axes[1]
ax.bar(lags_v, acf_vals_v, color='steelblue', alpha=0.7)
ax.axhline(ci_v, ls='--', color='red', lw=1, label='95% CI')
ax.axhline(-ci_v, ls='--', color='red', lw=1)
ax.axhline(0, color='black', lw=0.5)
ax.set_xlabel('Lag')
ax.set_ylabel('ACF')
ax.set_title('D3 — ACF of log V(d)')
ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig('d3_persistence.png', dpi=120)
plt.show()

# regression
res_ar = fit_ar_regression(log_v)
print('\nD3 AR regression (log V ~ const + lag1 + lag2):')
print(res_ar.summary2().tables[1][['Coef.','Std.Err.','t','P>|t|']].to_string())
print(f'R² = {res_ar.rsquared:.4f}')

print('\nD3 ACF of log V(d):')
for lag, val in zip(lags_v, acf_vals_v):
    sig = '*' if abs(val) > ci_v else ''
    print(f'  lag {lag:2d}: {val:+.4f}  {sig}')


# ══════════════════════════════════════════════════════════════════════════════
# D4. Within-session ACF of detrended residual
# ══════════════════════════════════════════════════════════════════════════════
u_acf, u_ci = pool_within_session_acf(bv_s, nlags=5)

fig, ax = plt.subplots(figsize=(6, 4))
lags_u = list(u_acf.keys())
vals_u = list(u_acf.values())
ax.bar(lags_u, vals_u, color='steelblue', alpha=0.7)
ax.axhline(u_ci, ls='--', color='red', lw=1, label='95% CI')
ax.axhline(-u_ci, ls='--', color='red', lw=1)
ax.axhline(0, color='black', lw=0.5)
ax.set_xlabel('Lag (bins within session)')
ax.set_ylabel('ACF')
ax.set_title('D4 — Within-session ACF of u(t,d)')
ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig('d4_acf_u.png', dpi=120)
plt.show()

print('\nD4 ACF of u(t,d) within session:')
for lag, val in u_acf.items():
    sig = '*' if abs(val) > u_ci else ''
    print(f'  lag {lag}: {val:+.4f}  {sig}')
print(f'95% CI bound: ±{u_ci:.4f}')


# ══════════════════════════════════════════════════════════════════════════════
# D5. Distribution of V(d)
# ══════════════════════════════════════════════════════════════════════════════
v_arr  = sess_tot.values
lv_arr = np.log(v_arr)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

axes[0].hist(v_arr, bins=25, color='steelblue', alpha=0.75, edgecolor='white')
axes[0].set_xlabel('V(d)')
axes[0].set_title('D5 — Histogram V(d)')

axes[1].hist(lv_arr, bins=25, color='steelblue', alpha=0.75, edgecolor='white')
axes[1].set_xlabel('log V(d)')
axes[1].set_title('D5 — Histogram log V(d)')

sm.qqplot(lv_arr, line='s', ax=axes[2], alpha=0.5)
axes[2].set_title('D5 — QQ plot log V(d) vs Normal')

plt.tight_layout()
plt.savefig('d5_distribution.png', dpi=120)
plt.show()

print('\nD5 V(d) stats:')
print(f'  mean={v_arr.mean():.0f}  std={v_arr.std():.0f}  skew={stats.skew(v_arr):.3f}  kurt={stats.kurtosis(v_arr):.3f}')
print('D5 log V(d) stats:')
print(f'  mean={lv_arr.mean():.4f}  std={lv_arr.std():.4f}  skew={stats.skew(lv_arr):.3f}  kurt={stats.kurtosis(lv_arr):.3f}')


# ══════════════════════════════════════════════════════════════════════════════
# D6. CV by bin
# ══════════════════════════════════════════════════════════════════════════════
g6 = bv.groupby('bin_idx')['Monto']
cv_vals = g6.std() / g6.mean()

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(np.arange(N_BINS), cv_vals.values, color='steelblue', alpha=0.75)
ax.set_xticks(np.arange(N_BINS))
ax.set_xticklabels(BIN_LABEL_STRS, rotation=45, ha='right', fontsize=8)
ax.set_ylabel('CV')
ax.set_title('D6 — Coefficient of Variation by bin')
plt.tight_layout()
plt.savefig('d6_cv.png', dpi=120)
plt.show()

print('\nD6 CV by bin:')
for label, cv in zip(BIN_LABEL_STRS, cv_vals.values):
    print(f'  {label}: {cv:.4f}')


# ══════════════════════════════════════════════════════════════════════════════
# D7. Outlier / regime sessions
# ══════════════════════════════════════════════════════════════════════════════
flags = flag_outlier_sessions(sess_tot, bv_s)
log_v_arr = np.log(sess_tot.values)
mu_lv = log_v_arr.mean()
std_lv = log_v_arr.std()

flagged_dates = set(flags['date'].dt.date if hasattr(flags['date'].iloc[0] if len(flags) else None, 'date') else [])

fig, ax = plt.subplots(figsize=(12, 4))
dates_all = sess_tot.index
ax.plot(range(len(dates_all)), log_v_arr, lw=1, color='steelblue', zorder=1)
ax.axhline(mu_lv + 2.5 * std_lv, ls='--', color='red', lw=1, label='±2.5σ')
ax.axhline(mu_lv - 2.5 * std_lv, ls='--', color='red', lw=1)

if len(flags) > 0:
    flag_v_idx = []
    for d in flags['date'].unique():
        matches = np.where(dates_all == d)[0]
        flag_v_idx.extend(matches.tolist())
    flag_v_idx = list(set(flag_v_idx))
    ax.scatter(flag_v_idx, log_v_arr[flag_v_idx], color='red', zorder=2, s=30, label='Flagged')

ax.set_xlabel('Session index')
ax.set_ylabel('log V(d)')
ax.set_title('D7 — log V(d) with outliers highlighted')
ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig('d7_outliers.png', dpi=120)
plt.show()

print('\nD7 Flagged sessions:')
print(flags.to_string(index=False) if len(flags) else '  none')


# ══════════════════════════════════════════════════════════════════════════════
# Summaries
# ══════════════════════════════════════════════════════════════════════════════
print_summary('D1 Summary',
    f'The average intraday volume profile across {len(sess_tot)} sessions shows that the shape is {shape_flag}. '
    f'The bin with the highest mean share is {prof.loc[prof["mean"].idxmax(), "bin_label"]} '
    f'(μ={prof["mean"].max():.4f}); the lowest is {prof.loc[prof["mean"].idxmin(), "bin_label"]} '
    f'(μ={prof["mean"].min():.4f}). The maximum deviation from uniform (1/18={flat_share:.4f}) '
    f'is {max_dev_flat:.4f}. Median and mean shares track closely across most bins, indicating '
    f'limited skew in the cross-sectional distribution.'
)

print_summary('D2 Summary',
    f'The diurnal profile is compared across weekdays and month-end/non-month-end splits. '
    f'The maximum absolute deviation between any two weekday profiles at any bin is {wd_max_dev:.4f}. '
    f'The maximum absolute deviation between month-end (day≥25) and non-month-end profiles is {me_max_dev:.4f}. '
    f'These values indicate the degree of shape variation by calendar grouping.'
)

acf_sig_lags_v = [int(l) for l, v in zip(lags_v, acf_vals_v) if abs(v) > ci_v]
print_summary('D3 Summary',
    f'log V(d) time series spans {len(log_v)} sessions. ACF is significant (|r|>1.96/√n={ci_v:.4f}) '
    f'at lags: {acf_sig_lags_v if acf_sig_lags_v else "none"}. '
    f'AR(2) regression: lag-1 coef={res_ar.params["lag1"]:.4f} (t={res_ar.tvalues["lag1"]:.2f}), '
    f'lag-2 coef={res_ar.params["lag2"]:.4f} (t={res_ar.tvalues["lag2"]:.2f}), R²={res_ar.rsquared:.4f}.'
)

u_sig_lags = [l for l, v in u_acf.items() if abs(v) > u_ci]
print_summary('D4 Summary',
    f'Within-session ACF of the detrended residual u(t,d) = s(t,d)/μ(t), pooled over all sessions '
    f'and respecting session boundaries. 95% CI bound = ±{u_ci:.4f}. '
    f'Significant lags: {u_sig_lags if u_sig_lags else "none"}. '
    f'ACF values: ' + ', '.join(f'lag {l}={v:+.4f}' for l, v in u_acf.items()) + '.'
)

print_summary('D5 Summary',
    f'V(d): mean={v_arr.mean():.0f}, std={v_arr.std():.0f}, skew={stats.skew(v_arr):.3f}, '
    f'excess_kurt={stats.kurtosis(v_arr):.3f}. '
    f'log V(d): mean={lv_arr.mean():.4f}, std={lv_arr.std():.4f}, skew={stats.skew(lv_arr):.3f}, '
    f'excess_kurt={stats.kurtosis(lv_arr):.3f}. '
    f'QQ plot shows degree of normality of log V(d) visually.'
)

print_summary('D6 Summary',
    f'CV(t) = std(v(t,d))/mean(v(t,d)) ranges from {cv_vals.min():.4f} ({BIN_LABEL_STRS[cv_vals.idxmin()]}) '
    f'to {cv_vals.max():.4f} ({BIN_LABEL_STRS[cv_vals.idxmax()]}). '
    f'Mean CV across bins: {cv_vals.mean():.4f}. '
    f'Bins with CV > 1.0: {[BIN_LABEL_STRS[i] for i in cv_vals[cv_vals > 1.0].index.tolist()]}.'
)

print_summary('D7 Summary',
    f'{len(flags)} flags raised across {flags["date"].nunique() if len(flags) else 0} sessions '
    f'(V outlier threshold: |z|>2.5 on log V; bin share threshold: >0.30). '
    f'Flagged dates: {flags["date"].unique().tolist() if len(flags) else "none"}.'
)